# Review Sentiment Analysis

This notebook predicts whether a review is negative, neutral, or positive.

**Input:** `data/amazon_reviews.csv` with columns `reviewText` and `overall`.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd().resolve()
if ROOT.name == 'case-study-4': ROOT = ROOT.parent
elif not (ROOT / 'case-study-4').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'case-study-4'))
from sentiment_analysis import read_reviews, split_data, train_lstm, LABEL_NAMES

# Ratings become labels: 1-2 negative, 3 neutral, 4-5 positive
reviews = read_reviews(ROOT / 'data' / 'amazon_reviews.csv', 'reviewText', 'overall', max_samples=3000)
train, valid, test = split_data(reviews)
print('Reviews:', len(reviews))
print('Train:', len(train), 'Validation:', len(valid), 'Test:', len(test))

In [ ]:
# Check the number of examples in each class
sns.countplot(data=reviews, x='label')
plt.xticks(range(3), LABEL_NAMES)
plt.title('Sentiment labels')
plt.show()

## Train the model

The BiLSTM reads the review text and predicts one of the three labels. One epoch is used so the notebook runs quickly.

In [ ]:
OUTPUT = ROOT / 'outputs' / 'case-study-4'
OUTPUT.mkdir(parents=True, exist_ok=True)
result = train_lstm(train, valid, test, OUTPUT, epochs=1)
print('Macro-F1:', round(result['macro_f1'], 3))
print('Weighted-F1:', round(result['weighted_f1'], 3))

## See the results

Rows are the real labels. Columns are the model predictions.

In [ ]:
sns.heatmap(np.array(result['confusion_matrix']), annot=True, fmt='d', cmap='Blues', cbar=False, xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

A higher Macro-F1 is better. Read some incorrect reviews as well, because ratings are only approximate sentiment labels.